<a href="https://colab.research.google.com/github/SikhuN-Ctrl/Datalyze/blob/main/DatalyzeMovieRecommendations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy==1.26.4 -q
!pip install scikit-surprise -q

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split as surprise_train_test_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

!wget -q http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q -o ml-100k.zip
# u.data: user id | item id | rating | timestamp  (tab-separated, no header)
ratings_df = pd.read_csv('ml-100k/u.data', sep='\t',
                          names=['user_id', 'item_id', 'rating', 'timestamp'])

# u.item: movie id | title | release date | video release date | IMDb URL | 19 binary genre flags
genre_names = ['unknown','Action','Adventure','Animation',"Children's",'Comedy','Crime',
               'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery',
               'Romance','Sci-Fi','Thriller','War','Western']
item_cols = ['item_id','title','release_date','video_release_date','imdb_url'] + genre_names
movies_df = pd.read_csv('ml-100k/u.item', sep='|', names=item_cols, encoding='latin-1')

# Build a space-separated genre string per movie, same role as 'description' before
def genres_to_text(row):
    return ' '.join([g for g in genre_names if row[g] == 1])

movies_df['description'] = movies_df.apply(genres_to_text, axis=1)
items_df = movies_df[['item_id', 'title', 'description']]

print(ratings_df.shape, items_df.shape)
ratings_df.head()

RELEVANCE_THRESHOLD = 4

train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)
print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
def popularity_recommendations(train_df, min_ratings=1, top_n=10):
    stats = train_df.groupby('item_id')['rating'].agg(['mean', 'count'])
    stats = stats[stats['count'] >= min_ratings]
    return stats.sort_values('mean', ascending=False).head(top_n)

popularity_recommendations(train_df, min_ratings=1)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(items_df['description'])
content_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
content_sim_df = pd.DataFrame(content_sim, index=items_df['item_id'], columns=items_df['item_id'])

def content_based_recommend(user_id, train_df, content_sim_df, top_n=10):
    user_ratings = train_df[train_df['user_id'] == user_id]
    liked_items = user_ratings[user_ratings['rating'] >= RELEVANCE_THRESHOLD]['item_id']
    if liked_items.empty:
        return pd.Series(dtype=float)
    scores = content_sim_df[liked_items].mean(axis=1)
    scores = scores.drop(index=user_ratings['item_id'], errors='ignore')
    return scores.sort_values(ascending=False).head(top_n)

content_based_recommend(user_id=1, train_df=train_df, content_sim_df=content_sim_df)
reader = Reader(rating_scale=(ratings_df['rating'].min(), ratings_df['rating'].max()))
surprise_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
trainset = surprise_data.build_full_trainset()

# Item-item KNN
knn_model = KNNBasic(sim_options={'user_based': False}, verbose=False)
knn_model.fit(trainset)

# SVD (matrix factorization)
svd_model = SVD(random_state=42)
svd_model.fit(trainset)

def cf_predict(model, user_id, item_id):
    return model.predict(user_id, item_id).est

# Example prediction
print("KNN prediction:", cf_predict(knn_model, 1, 40))
print("SVD prediction:", cf_predict(svd_model, 1, 40))
def hybrid_score(user_id, item_id, train_df, content_sim_df, cf_model, alpha=0.5):
    # Content-based component: avg similarity to items the user liked
    user_ratings = train_df[train_df['user_id'] == user_id]
    liked_items = user_ratings[user_ratings['rating'] >= RELEVANCE_THRESHOLD]['item_id']
    if len(liked_items) > 0 and item_id in content_sim_df.index:
        content_score = content_sim_df.loc[item_id, liked_items].mean()
        # scale similarity (0-1) onto rating scale for blending
        content_score_scaled = content_score * ratings_df['rating'].max()
    else:
        content_score_scaled = ratings_df['rating'].mean()

    cf_score = cf_predict(cf_model, user_id, item_id)

    return alpha * content_score_scaled + (1 - alpha) * cf_score

# Example
hybrid_score(user_id=1, item_id=40, train_df=train_df, content_sim_df=content_sim_df, cf_model=svd_model, alpha=0.5)
def evaluate_rating_predictions(predict_fn, test_df):
    y_true, y_pred = [], []
    for _, row in test_df.iterrows():
        y_true.append(row['rating'])
        y_pred.append(predict_fn(row['user_id'], row['item_id']))
    rmse = mean_squared_error(y_true, y_pred)**0.5
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

def precision_recall_f1_at_k(predict_fn, test_df, k=10, threshold=RELEVANCE_THRESHOLD):
    precisions, recalls = [], []
    for user_id in test_df['user_id'].unique():
        user_test = test_df[test_df['user_id'] == user_id]
        preds = [(row['item_id'], predict_fn(user_id, row['item_id']), row['rating'])
                 for _, row in user_test.iterrows()]
        preds.sort(key=lambda x: x[1], reverse=True)
        top_k = preds[:k]

        n_relevant = sum(1 for _, _, true_r in preds if true_r >= threshold)
        n_rec_relevant = sum(1 for _, _, true_r in top_k if true_r >= threshold)

        precisions.append(n_rec_relevant / len(top_k) if top_k else 0)
        recalls.append(n_rec_relevant / n_relevant if n_relevant else 0)

    precision = np.mean(precisions)
    recall = np.mean(recalls)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# --- Wire up predict functions for each model ---
knn_predict = lambda u, i: cf_predict(knn_model, u, i)
svd_predict = lambda u, i: cf_predict(svd_model, u, i)
hybrid_predict = lambda u, i: hybrid_score(u, i, train_df, content_sim_df, svd_model, alpha=0.5)

results = []
for name, fn in [('KNN (item-item)', knn_predict), ('SVD', svd_predict), ('Hybrid', hybrid_predict)]:
    rmse, mae = evaluate_rating_predictions(fn, test_df)
    prec, rec, f1 = precision_recall_f1_at_k(fn, test_df, k=10)
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'Precision@10': prec, 'Recall@10': rec, 'F1@10': f1})

comparison_df = pd.DataFrame(results)
comparison_df

(100000, 4) (1682, 3)
Train: 80000 rows | Test: 20000 rows
KNN prediction: 3.1716296513567914
SVD prediction: 3.108715366572673


,Model,RMSE,MAE,Precision@10,Recall@10,F1@10
0,KNN (item-item),0.972777,0.767993,0.666568,0.710134,0.687662
1,SVD,0.934634,0.737631,0.669547,0.711518,0.689895
2,Hybrid,1.578564,1.369754,0.654866,0.702600,0.677893
